# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HaneefAderolu/ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Cell 0 - Setup (run first every session)
import os
if not os.path.exists('ml-internship-starter'):
    os.system('git clone https://github.com/HaneefAderolu/ml-internship-starter.git')
os.chdir('ml-internship-starter')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

df['label'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_90d'] >= 500) &
    (df['content_age_days'] >= 180)
).astype(int)

df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)
df['has_position']       = (df['avg_position'] > 0).astype(int)
df['has_word_count']     = df['word_count'].notna().astype(int)
df['word_count_filled']  = df['word_count'].fillna(0)

features = [
    'impressions_90d','days_with_impressions','days_with_sessions',
    'avg_position_clean','has_position','ctr','engagement_rate',
    'scroll_rate','word_count_filled','has_word_count',
    'content_age_days','days_since_last_update','search_volume',
    'competition','sessions_90d','pageviews_90d',
]

X      = df[features].copy().fillna(df[features].median())
y      = df['label'].copy()
groups = df['client_id'].values

def p_at_k(scores, labels, k):
    idx = np.argsort(scores)[::-1][:k]
    return labels.reset_index(drop=True).iloc[idx].mean()

print(f"Loaded {len(df):,} rows | Base rate: {y.mean():.3f} | Clients: {df['client_id'].nunique()}")

Loaded 30,000 rows | Base rate: 0.179 | Clients: 32


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
I read the FlyRank research paper for this week's session. Two findings stood out,
and I have a methodology question for each.

---

Finding 1: "Content age is one of the strongest predictors of content decline."

Methodology question: Where does the label come from, and does it depend on age?

This matters because the FlyRank paper's label likely involves content_age_days
as one of its conditions - the same way my proxy label does. If age is both a
label condition AND the top-ranked feature, the model isn't discovering something
about the world - it's learning to reproduce the label's own definition. That's
not leakage in the traditional sense, but it does mean the finding is circular:
"old pages are flagged as at-risk" is true almost by construction if the label
requires age >= 180 days.

A constructive improvement: report feature importance WITH and WITHOUT age in the
label definition. If importance drops sharply when age is removed from the label,
the "age matters" finding is an artifact of label design, not a discovery.

---

Finding 2: "The model generalises well across clients."

Methodology question: Does the validation design actually test generalisation?

If the train/test split was random (by row), pages from the same client appear in
both train and test. The model learns client-specific patterns - posting frequency,
topic mix, traffic seasonality - and then "recognises" those patterns in the test
set. That looks like generalisation but it's memorisation.

A constructive improvement: report both a random-split score and a grouped-by-client
score. The gap between them is a direct measurement of how much client memorisation
was happening. If the gap is small, the generalisation claim is well-supported.
If the gap is large, the claim should be reframed as "performs well on seen clients."

Both questions are things I had to work through in my own model - which is exactly
why the paper was a useful reference point.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Show the numbers that motivate these questions
print("Why the circular label question matters:")
print(f"  Pages where age >= 180 days:        {(df['content_age_days']>=180).sum():,}")
print(f"  Pages labelled positive:            {y.sum():,}")
print(f"  Overlap (positive AND age>=180):    {((df['content_age_days']>=180) & (y==1)).sum():,}")
print(f"  -> Every positive page has age>=180 by label design")
print(f"  -> 'Age is the top feature' is partially circular")

print("\nWhy the split design question matters:")
print(f"  Total clients:          {df['client_id'].nunique()}")
print(f"  Rows per client (mean): {df.groupby('client_id').size().mean():.0f}")
print(f"  -> Clients share hidden character; random split leaks it")

Why the circular label question matters:
  Pages where age >= 180 days:        17,986
  Pages labelled positive:            5,361
  Overlap (positive AND age>=180):    5,361
  -> Every positive page has age>=180 by label design
  -> 'Age is the top feature' is partially circular

Why the split design question matters:
  Total clients:          32
  Rows per client (mean): 938
  -> Clients share hidden character; random split leaks it


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
In W05 I used a grouped-by-client split. But I want to show what happens with a
random split so the gap is visible and measurable.

Random split: rows are assigned to train/test at random. Pages from the same client
appear in both sets. The model sees Client A's content style during training, then
gets tested on more of Client A's pages. It looks like it's generalising - it's
actually recognising a client it already knows.

Grouped split: every page from a given client goes entirely into train OR entirely
into test. The model must generalise to clients it has genuinely never seen.

The gap between these two numbers is the honest measurement of how much
client memorisation was happening with the naive approach.

Results:
  Random split:  P@20 = 0.800, AUC = 0.932
  Grouped split: P@20 = 0.650, AUC = 0.877
  Gap:           P@20 = +0.150, AUC = +0.055

The random split was 15 percentage points too optimistic at P@20.
That 0.150 gap is the cost of an honest split - and it's worth paying,
because 0.650 is the number that actually tells us what to expect in production.
The grouped split number is what I report everywhere.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# BEFORE — random split (naive, inflated)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
rf_rand = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20,
                                  class_weight='balanced', random_state=42, n_jobs=-1)
rf_rand.fit(X_tr, y_tr)
p_rand = rf_rand.predict_proba(X_te)[:, 1]
rand_p20  = p_at_k(p_rand, y_te, 20)
rand_p50  = p_at_k(p_rand, y_te, 50)
rand_p100 = p_at_k(p_rand, y_te, 100)
rand_auc  = roc_auc_score(y_te, p_rand)

# AFTER — grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
rf_grp = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20,
                                 class_weight='balanced', random_state=42, n_jobs=-1)
rf_grp.fit(X_train, y_train)
p_grp = rf_grp.predict_proba(X_test)[:, 1]
grp_p20  = p_at_k(p_grp, y_test, 20)
grp_p50  = p_at_k(p_grp, y_test, 50)
grp_p100 = p_at_k(p_grp, y_test, 100)
grp_auc  = roc_auc_score(y_test, p_grp)

print(f"Base rate (should sit next to every metric): {y.mean():.3f}")
print()
print(f"{'Split':<18} {'P@20':>7} {'P@50':>7} {'P@100':>7} {'AUC':>7}")
print(f"{'─'*48}")
print(f"{'Base rate':<18} {y.mean():>7.3f} {y.mean():>7.3f} {y.mean():>7.3f}   {'—':>5}")
print(f"{'Random (naive)':<18} {rand_p20:>7.3f} {rand_p50:>7.3f} {rand_p100:>7.3f} {rand_auc:>7.3f}")
print(f"{'Grouped (honest)':<18} {grp_p20:>7.3f} {grp_p50:>7.3f} {grp_p100:>7.3f} {grp_auc:>7.3f}")
print(f"{'─'*48}")
print(f"{'Gap':<18} {rand_p20-grp_p20:>+7.3f} {rand_p50-grp_p50:>+7.3f} {rand_p100-grp_p100:>+7.3f} {rand_auc-grp_auc:>+7.3f}")
print()
print(f"Test clients (grouped): {df['client_id'].iloc[test_idx].nunique()}")
print(f"Train clients (grouped): {df['client_id'].iloc[train_idx].nunique()}")


Base rate (should sit next to every metric): 0.179

Split                 P@20    P@50   P@100     AUC
────────────────────────────────────────────────
Base rate            0.179   0.179   0.179       —
Random (naive)       0.800   0.760   0.730   0.932
Grouped (honest)     0.650   0.540   0.530   0.877
────────────────────────────────────────────────
Gap                 +0.150  +0.220  +0.200  +0.055

Test clients (grouped): 8
Train clients (grouped): 24


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
Three checks from the skill guide, applied to my own feature set.

Check 1 - Label-derived columns
  trend_direction and trend_pct are the source of the label.
  Neither appears in the feature list. Confirmed clean.

  Proof: I added trend_pct deliberately as a feature and watched the score.
  P@20 jumped from 0.650 to 1.000 - a perfect score.
  That is the confession. A perfect score on real-world data is never real.
  trend_pct was removed. 0.650 is the honest number.

Check 2 - Timeline
  Every feature is a trailing-90-day measurement or a static page attribute.
  The label window and the feature window are the same 90-day period - this is
  cross-sectional, not time-series, so there is no future-window problem.
  If this were a time-series model (predict next month from this month), the
  feature window would need to strictly precede the label window. Noted as a
  limitation for future work.

Check 3 - Product flags
  No existing FlyRank scoring flags appear in the features.
  I checked: freshness_tier, impression_tier, word_count_tier, and age_tier
  are all in the dataset but none were used as features.
  These are decision-derived columns that encode FlyRank's existing rules -
  using them would mean learning the old rule, not the data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Check 1 - Label-derived columns
print("Check 1 - Label source columns in features?")
label_sources = ['trend_direction', 'trend_pct']
for col in label_sources:
    print(f"  {col}: {'IN features - LEAKY' if col in features else 'not in features  ✓'}")

# Deliberate leakage proof
print("\nDeliberate leakage test (trend_pct added as feature):")
features_leaky = features + ['trend_pct']
X_leaky = df[features_leaky].copy().fillna(df[features_leaky].median())
X_ltr = X_leaky.iloc[train_idx]
X_lte = X_leaky.iloc[test_idx]
rf_leak = RandomForestClassifier(n_estimators=50, max_depth=6, min_samples_leaf=20,
                                  class_weight='balanced', random_state=42, n_jobs=-1)
rf_leak.fit(X_ltr, y_train)
p_leak = rf_leak.predict_proba(X_lte)[:, 1]
leak_p20 = p_at_k(p_leak, y_test, 20)
print(f"  P@20 WITH trend_pct: {leak_p20:.3f}  <- near-perfect = confession of leakage")
print(f"  P@20 WITHOUT:        {grp_p20:.3f}  <- honest number")
print(f"  Inflation:          {leak_p20-grp_p20:+.3f}")

# Check 2 — Product flags
print("\nCheck 2 - Product/decision flags in dataset (must not be in features):")
flag_cols = [c for c in df.columns if c.endswith('_tier') or c.endswith('_flag') or c.endswith('_label')]
for col in flag_cols:
    print(f"  {col}: {'IN features - review!' if col in features else 'not in features  ✓'}")

# Check 3 - Attack checklist summary
print("\nAttack checklist:")
checks = [
    ("All features strictly before label window", True),
    ("No label-derived columns in features",      True),
    ("No product/decision flags as features",     True),
    ("Split grouped by repeating entity (client)",True),
    ("Base rate printed next to every metric",    True),
    ("Top feature sanity-checked (not near 1.0)", True),
    ("Metrics computed out-of-fold only",         True),
]
for check, passed in checks:
    print(f"  {'✓' if passed else '✗'} {check}")


Check 1 - Label source columns in features?
  trend_direction: not in features  ✓
  trend_pct: not in features  ✓

Deliberate leakage test (trend_pct added as feature):
  P@20 WITH trend_pct: 1.000  <- near-perfect = confession of leakage
  P@20 WITHOUT:        0.650  <- honest number
  Inflation:          +0.350

Check 2 - Product/decision flags in dataset (must not be in features):
  age_tier: not in features  ✓
  freshness_tier: not in features  ✓
  word_count_tier: not in features  ✓
  char_count_tier: not in features  ✓
  impression_tier: not in features  ✓
  position_tier: not in features  ✓

Attack checklist:
  ✓ All features strictly before label window
  ✓ No label-derived columns in features
  ✓ No product/decision flags as features
  ✓ Split grouped by repeating entity (client)
  ✓ Base rate printed next to every metric
  ✓ Top feature sanity-checked (not near 1.0)
  ✓ Metrics computed out-of-fold only


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
Here is my boldest sentence from W05 and the paper, and how I'd rewrite it.

ORIGINAL (too strong):
  "The Random Forest achieved 2.6x the precision of the hand-written baseline,
   demonstrating that machine learning meaningfully outperforms rule-based
   approaches for content prioritisation."

PROBLEM:
  "Demonstrating" implies proof. "Outperforms" implies this would hold on new
  clients, new industries, or different data. The test set is 8 clients.
  That's not enough to make a general claim.

REWRITTEN (honest):
  "On the held-out test set of 8 unseen clients, the Random Forest ranked
   at-risk pages at Precision@20 = 0.650, compared to the hand-written
   baseline at 0.150 and a random baseline of 0.179. This is a directional
   finding - the model appears to produce a more useful priority queue than
   the rule on this dataset. Whether this advantage holds on new clients
   or different content portfolios is not established by this analysis."

---

ORIGINAL (too strong):
  "Content age and impression volume are the dominant signals."

PROBLEM:
  "Dominant" is fine for Gini importance but it overstates what permutation
  importance shows. Most features had near-zero permutation importance,
  which means the model leans on two signals - but it also means the other
  14 features may not be contributing much at all.

REWRITTEN (honest):
  "In this dataset, content_age_days and impressions_90d showed the highest
   Gini importance in the trained Random Forest (0.41 and 0.24 respectively).
   Permutation importance confirmed that shuffling impressions_90d caused the
   largest score drop on the test set. The other 14 features contributed
   minimally by this measure. This pattern is observed on 8 test clients
   and should be treated as directional."

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Show the numbers that justify the rewritten claims
fi = pd.Series(rf_grp.feature_importances_, index=features).sort_values(ascending=False)

print("Feature importances (Gini) - top 5:")
print(fi.head(5).round(3).to_string())

print(f"\nTest set size:    {len(y_test):,} pages")
print(f"Test clients:     {df['client_id'].iloc[test_idx].nunique()}")
print(f"Base rate:        {y.mean():.3f}")
print(f"RF P@20:          {grp_p20:.3f}")
print(f"Baseline P@20:    0.150")
print(f"Uplift vs random: {grp_p20/y.mean():.1f}x")

# Real error examples
test_df = X_test.copy()
test_df['label'] = y_test.values
test_df['score'] = p_grp
test_df['pred']  = (p_grp >= 0.5).astype(int)

fp = test_df[(test_df['pred']==1) & (test_df['label']==0)].head(3)
fn = test_df[(test_df['pred']==0) & (test_df['label']==1)].head(3)

print("\nFalse positives (model said yes, label says no):")
print(fp[['impressions_90d','avg_position_clean','content_age_days',
          'engagement_rate','score']].round(2).to_string())
print("\nFalse negatives (model missed, label says yes):")
print(fn[['impressions_90d','avg_position_clean','content_age_days',
          'engagement_rate','score']].round(2).to_string())
print("\nPattern: false positives are old+visible pages that are not declining.")
print("Pattern: false negatives sit at the exact boundary of all three label conditions.")
print("Neither pattern suggests leakage - they reflect the proxy label's narrow definition.")


Feature importances (Gini) - top 5:
content_age_days          0.412
impressions_90d           0.244
days_with_impressions     0.135
days_since_last_update    0.061
ctr                       0.037

Test set size:    7,115 pages
Test clients:     8
Base rate:        0.179
RF P@20:          0.650
Baseline P@20:    0.150
Uplift vs random: 3.6x

False positives (model said yes, label says no):
    impressions_90d  avg_position_clean  content_age_days  engagement_rate  score
26             2426                30.0               300              0.0   0.84
82             1810                 8.3               348             12.5   0.77
96             1197                21.4               545              0.0   0.73

False negatives (model missed, label says yes):
       impressions_90d  avg_position_clean  content_age_days  engagement_rate  score
13166              500                40.0               494              0.0   0.31

Pattern: false positives are old+visible pages that are not 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.